# Full Backtest — all active symbols, deployed configs

**Date:** 2026-04-20  
**Active universe:** BTC · HYPE · ZEC · XRP · kPEPE · FARTCOIN · LIT · ENA · SOL (9 symbols, SILVER + ETH demoted)

Runs each symbol's `config/deployed/whale_<SYMBOL>.json` exactly as the bot uses it. Includes BTC-1h-confirm (ON for 6 symbols), 4h filter (where enabled), multi-tier TP1/TP2/TP3 scale-out.

Output: per-symbol metrics, aggregate P&L equity curve, walk-forward split, exit-reason breakdown, regime split.

In [ ]:
import os, sys, json, time
from dataclasses import dataclass
from collections import defaultdict

sys.path.insert(0, '/Users/lucaneto/Trading/shared')
sys.path.insert(0, '/Users/lucaneto/Trading/swing-trading-bot')

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')
plt.rcParams.update({
    'figure.facecolor': '#0b0f0e', 'axes.facecolor': '#0b0f0e',
    'axes.edgecolor': '#1b2420', 'axes.labelcolor': '#e6eeea',
    'xtick.color': '#7a8480', 'ytick.color': '#7a8480',
    'text.color': '#e6eeea', 'axes.titlecolor': '#e6eeea',
    'grid.color': '#1b2420', 'legend.facecolor': '#0b0f0e',
    'legend.edgecolor': '#1b2420'
})

SYMBOLS = ['BTC', 'HYPE', 'ZEC', 'XRP', 'kPEPE', 'FARTCOIN', 'LIT', 'ENA', 'SOL']
HL_CAP = {'BTC': 40, 'HYPE': 10, 'ZEC': 10, 'XRP': 20, 'kPEPE': 10,
          'FARTCOIN': 10, 'LIT': 5, 'ENA': 10, 'SOL': 20}
HL_API = 'https://api.hyperliquid.xyz/info'
COMMISSION = 0.00006
CACHE_DIR = '/Users/lucaneto/Trading/commodities-bot/research/notebooks/data/whale_swing_cache'
DEPLOY_DIR = '/Users/lucaneto/Trading/swing-trading-bot/config/deployed'

In [ ]:
# ── helpers (fetch + features + backtest) ──
def fetch_hl(symbol, interval, bars=10000):
    cache_file = os.path.join(CACHE_DIR, f'{symbol.replace(":", "_")}_{interval}.csv')
    if os.path.exists(cache_file):
        age = time.time() - os.path.getmtime(cache_file)
        if age < 3600 * 4:
            return pd.read_csv(cache_file, parse_dates=['timestamp'])
    ms = {'5m': 300_000, '1h': 3_600_000, '4h': 14_400_000}[interval]
    end = int(time.time() * 1000)
    all_data, remaining, CHUNK = [], bars, 4500
    while remaining > 0:
        take = min(CHUNK, remaining)
        start = end - ms * take
        r = requests.post(HL_API, json={'type': 'candleSnapshot',
            'req': {'coin': symbol, 'interval': interval, 'startTime': start, 'endTime': end}}, timeout=30)
        data = r.json() or []
        if not data: break
        all_data = data + all_data
        end = int(data[0]['t']) - ms
        remaining -= len(data)
        if len(data) < take * 0.5: break
        time.sleep(0.3)
    seen = {int(c['t']): c for c in all_data}
    sd = sorted(seen.values(), key=lambda c: int(c['t']))
    df = pd.DataFrame([{
        'timestamp': pd.to_datetime(int(c['t']), unit='ms', utc=True),
        'open': float(c['o']), 'high': float(c['h']),
        'low': float(c['l']), 'close': float(c['c']),
        'volume': float(c['v']),
    } for c in sd])
    df.to_csv(cache_file, index=False)
    return df

def add_features(df):
    df = df.copy()
    c = df['close']; h = df['high']; l = df['low']
    df['ema_21'] = c.ewm(span=21, adjust=False).mean()
    df['ema_50'] = c.ewm(span=50, adjust=False).mean()
    df['ema_200'] = c.ewm(span=200, adjust=False).mean()
    df['ema_50_slope'] = (df['ema_50'] - df['ema_50'].shift(20)) / df['ema_50'].shift(20)
    delta = c.diff()
    gain = delta.where(delta > 0, 0.0).ewm(alpha=1/14, adjust=False).mean()
    loss = (-delta.where(delta < 0, 0.0)).ewm(alpha=1/14, adjust=False).mean()
    rs = gain / loss.replace(0, np.nan)
    df['rsi'] = 100 - 100 / (1 + rs)
    tr = pd.concat([h-l, (h-c.shift(1)).abs(), (l-c.shift(1)).abs()], axis=1).max(axis=1)
    df['atr'] = tr.ewm(alpha=1/14, adjust=False).mean()
    df['bb_mid'] = c.rolling(20).mean()
    bb_std = c.rolling(20).std()
    df['bb_upper'] = df['bb_mid'] + 2 * bb_std
    df['bb_lower'] = df['bb_mid'] - 2 * bb_std
    return df

from core.features import trend_lookup_1h, structure_lookup_1h

@dataclass
class Cfg:
    trend_filter: str; entry_type: str
    rsi_oversold: float; rsi_overbought: float
    sl_atr: float; tp1_atr: float; tp1_pct: float; trail_atr: float
    max_hold_bars: int; direction: str; use_1h_filter: bool
    trend_filter_1h: str = 'both_agree'
    require_4h_agreement: bool = False
    require_btc_1h_confirm: bool = False
    tp2_atr: float = 0.0; tp2_pct: float = 0.0
    tp3_atr: float = 0.0; tp3_pct: float = 0.0

def build_btc_confirm(df5, df1h_btc):
    btc = df1h_btc.copy()
    btc['ret'] = np.log(btc['close']).diff()
    btc_ts = btc['timestamp'].values; btc_ret = btc['ret'].values
    t5 = df5['timestamp'].values
    out = np.zeros(len(t5), dtype=np.int8); j = 0
    for i in range(len(t5)):
        while j + 1 < len(btc_ts) and btc_ts[j + 1] <= t5[i]: j += 1
        r = btc_ret[j] if j < len(btc_ret) else np.nan
        out[i] = 0 if (np.isnan(r) or r == 0) else (1 if r > 0 else -1)
    return out

def precompute(df5, df1h, df4h, btc_confirm):
    a = {col: df5[col].to_numpy() for col in
         ['open','high','low','close','atr','rsi','ema_21','ema_50','ema_200',
          'ema_50_slope','bb_lower','bb_upper']}
    up_e, dn_e = trend_lookup_1h(df5, df1h)
    up_s, dn_s = structure_lookup_1h(df5, df1h)
    up_4h, dn_4h = structure_lookup_1h(df5, df4h, pivot_bars=3)
    a.update({'up_1h': up_e, 'dn_1h': dn_e, 'up_struct': up_s, 'dn_struct': dn_s,
              'up_4h': up_4h, 'dn_4h': dn_4h, 'btc_dir': btc_confirm,
              'timestamp': df5['timestamp'].values})
    return a

def backtest(arr, cfg, leverage, require_btc_confirm=False, i_start=52, i_end=None):
    n = len(arr['close']); i_end = i_end or n
    close=arr['close']; high=arr['high']; low=arr['low']
    rsi=arr['rsi']; atr=arr['atr']
    e21=arr['ema_21']; e50=arr['ema_50']; e200=arr['ema_200']; slope=arr['ema_50_slope']
    bbl=arr['bb_lower']; bbu=arr['bb_upper']
    btc_dir = arr['btc_dir']; ts = arr['timestamp']
    if cfg.trend_filter_1h == 'structure': up_1h = arr['up_struct']; dn_1h = arr['dn_struct']
    elif cfg.trend_filter_1h == 'both_agree': up_1h = arr['up_1h'] & arr['up_struct']; dn_1h = arr['dn_1h'] & arr['dn_struct']
    else: up_1h = arr['up_1h']; dn_1h = arr['dn_1h']
    trades = []; position = None; tp1_hit = tp2_hit = tp3_hit = False
    for i in range(max(i_start, 52), i_end):
        a_i = atr[i]; r = rsi[i]
        if a_i <= 0 or a_i != a_i or r != r: continue
        r_prev = rsi[i-1]; price = close[i]; hi = high[i]; lo = low[i]
        if position is not None:
            side = position['side']; entry = position['entry']; trl = position['trail_offset']
            if trl > 0:
                if side == 'long':
                    if hi > position['best']: position['best'] = hi
                    if not position['trail_active'] and hi >= entry + trl: position['trail_active'] = True
                    if position['trail_active']:
                        ns = position['best'] - trl
                        if ns > position['sl']: position['sl'] = ns
                else:
                    if lo < position['best']: position['best'] = lo
                    if not position['trail_active'] and lo <= entry - trl: position['trail_active'] = True
                    if position['trail_active']:
                        ns = position['best'] + trl
                        if ns < position['sl']: position['sl'] = ns
            # TP1/TP2/TP3
            if not tp1_hit and position['tp1'] is not None:
                tp1 = position['tp1']
                if (side == 'long' and hi >= tp1) or (side == 'short' and lo <= tp1):
                    tp1_hit = True; pct = cfg.tp1_pct
                    pnl_p = ((tp1-entry) if side == 'long' else (entry-tp1)) * position['size'] * pct
                    pnl_p -= position['notional'] * pct * COMMISSION * 2
                    trades.append({'pnl': pnl_p, 'side': side, 'reason': 'tp1', 'ts': ts[i], 'btc': int(position['btc_at_entry'])})
                    position['size'] *= (1-pct); position['notional'] *= (1-pct); position['sl'] = entry
            if tp1_hit and not tp2_hit and position.get('tp2') is not None:
                tp2 = position['tp2']
                if (side == 'long' and hi >= tp2) or (side == 'short' and lo <= tp2):
                    tp2_hit = True; pct = cfg.tp2_pct
                    pnl_p = ((tp2-entry) if side == 'long' else (entry-tp2)) * position['size'] * pct
                    pnl_p -= position['notional'] * pct * COMMISSION * 2
                    trades.append({'pnl': pnl_p, 'side': side, 'reason': 'tp2', 'ts': ts[i], 'btc': int(position['btc_at_entry'])})
                    position['size'] *= (1-pct); position['notional'] *= (1-pct)
            if tp2_hit and not tp3_hit and position.get('tp3') is not None:
                tp3 = position['tp3']
                if (side == 'long' and hi >= tp3) or (side == 'short' and lo <= tp3):
                    tp3_hit = True; pct = cfg.tp3_pct
                    pnl_p = ((tp3-entry) if side == 'long' else (entry-tp3)) * position['size'] * pct
                    pnl_p -= position['notional'] * pct * COMMISSION * 2
                    trades.append({'pnl': pnl_p, 'side': side, 'reason': 'tp3', 'ts': ts[i], 'btc': int(position['btc_at_entry'])})
                    position['size'] *= (1-pct); position['notional'] *= (1-pct)
            sl_hit = (lo <= position['sl']) if side == 'long' else (hi >= position['sl'])
            max_hold_hit = (i - position['entry_bar']) >= cfg.max_hold_bars
            if sl_hit or max_hold_hit:
                ep = position['sl'] if sl_hit else price
                pnl = ((ep-entry) if side == 'long' else (entry-ep)) * position['size']
                pnl -= position['notional'] * COMMISSION * 2
                reason = 'trail_stop' if (position['trail_active'] and sl_hit) else ('max_hold' if max_hold_hit else 'sl')
                trades.append({'pnl': pnl, 'side': side, 'reason': reason, 'ts': ts[i], 'btc': int(position['btc_at_entry'])})
                position = None; tp1_hit = tp2_hit = tp3_hit = False
                continue
        if position is not None: continue
        tf = cfg.trend_filter
        if tf == 'ema_cross': up_ok = e21[i] > e50[i]; dn_ok = e21[i] < e50[i]
        elif tf == 'ema_slope': up_ok = slope[i] > 0; dn_ok = slope[i] < 0
        elif tf == 'ema200':
            up_ok = price > e200[i] if e200[i] > 0 else True
            dn_ok = price < e200[i] if e200[i] > 0 else True
        else: up_ok = True; dn_ok = True
        if cfg.use_1h_filter:
            if not up_1h[i]: up_ok = False
            if not dn_1h[i]: dn_ok = False
        if cfg.require_4h_agreement:
            if not arr['up_4h'][i]: up_ok = False
            if not arr['dn_4h'][i]: dn_ok = False
        d = cfg.direction
        if d == 'long_only': dn_ok = False
        elif d == 'short_only': up_ok = False
        long_trig = short_trig = False; et = cfg.entry_type
        if et == 'rsi_bounce':
            long_trig = up_ok and r_prev < cfg.rsi_oversold and r >= cfg.rsi_oversold
            short_trig = dn_ok and r_prev > cfg.rsi_overbought and r <= cfg.rsi_overbought
        elif et == 'bb_touch':
            long_trig = up_ok and low[i-1] <= bbl[i] and price > high[i-1]
            short_trig = dn_ok and high[i-1] >= bbu[i] and price < low[i-1]
        elif et == 'ema_bounce':
            long_trig = up_ok and low[i-1] <= e21[i] and price > e21[i] and r_prev < 45
            short_trig = dn_ok and high[i-1] >= e21[i] and price < e21[i] and r_prev > 55
        elif et == 'swing_pivot':
            if i >= 55:
                if up_ok and low[i-3] < low[i-4] and low[i-3] < low[i-2] and price > high[i-1]: long_trig = True
                if dn_ok and high[i-3] > high[i-4] and high[i-3] > high[i-2] and price < low[i-1]: short_trig = True
        if require_btc_confirm:
            if long_trig and btc_dir[i] != 1: long_trig = False
            if short_trig and btc_dir[i] != -1: short_trig = False
        if long_trig or short_trig:
            side = 'long' if long_trig else 'short'
            margin = 10000 * 0.15; notional = margin * leverage; size = notional / price
            sl = price - a_i * cfg.sl_atr if side == 'long' else price + a_i * cfg.sl_atr
            tp1 = (price + a_i * cfg.tp1_atr if side == 'long' else price - a_i * cfg.tp1_atr) if cfg.tp1_atr > 0 else None
            tp2 = (price + a_i * cfg.tp2_atr if side == 'long' else price - a_i * cfg.tp2_atr) if cfg.tp2_atr > 0 else None
            tp3 = (price + a_i * cfg.tp3_atr if side == 'long' else price - a_i * cfg.tp3_atr) if cfg.tp3_atr > 0 else None
            trail = a_i * cfg.trail_atr if cfg.trail_atr > 0 else 0.0
            position = dict(entry=price, side=side, size=size, notional=notional,
                           sl=sl, tp1=tp1, tp2=tp2, tp3=tp3,
                           trail_offset=trail, trail_active=False, best=price, entry_bar=i,
                           btc_at_entry=btc_dir[i])
            tp1_hit = tp2_hit = tp3_hit = False
    return trades

print('helpers ready')

## 1 — Fetch + run deployed configs for all 9 symbols

In [ ]:
print('Fetching BTC 1h (shared by BTC-confirm array)...')
df1h_btc = fetch_hl('BTC', '1h', 2000)

results = {}
all_trades = []
for sym in SYMBOLS:
    fn = f'whale_{sym.replace(":","_")}.json'
    d = json.load(open(os.path.join(DEPLOY_DIR, fn)))
    cfg_d = Cfg(
        trend_filter=d['trend_filter'], entry_type=d['entry_type'],
        rsi_oversold=d['rsi_oversold'], rsi_overbought=d['rsi_overbought'],
        sl_atr=d['sl_atr'], tp1_atr=d['tp1_atr'], tp1_pct=d['tp1_pct'],
        trail_atr=d['trail_atr'], max_hold_bars=d['max_hold_bars'],
        direction=d['direction'], use_1h_filter=d['use_1h_filter'],
        trend_filter_1h=d.get('trend_filter_1h', 'ema_cross'),
        require_4h_agreement=bool(d.get('require_4h_agreement', False)),
        require_btc_1h_confirm=bool(d.get('require_btc_1h_confirm', False)),
        tp2_atr=float(d.get('tp2_atr', 0.0)), tp2_pct=float(d.get('tp2_pct', 0.0)),
        tp3_atr=float(d.get('tp3_atr', 0.0)), tp3_pct=float(d.get('tp3_pct', 0.0)),
    )
    df5 = add_features(fetch_hl(sym, '5m', 10000))
    df1h = add_features(fetch_hl(sym, '1h', 2000))
    df4h = add_features(fetch_hl(sym, '4h', 1000))
    if len(df5) < 250:
        print(f'  {sym}: only {len(df5)} bars, skip'); continue
    arr = precompute(df5, df1h, df4h, build_btc_confirm(df5, df1h_btc))
    eff_lev = HL_CAP[sym] * 0.15
    n = len(df5); mid = n // 2

    full = backtest(arr, cfg_d, leverage=eff_lev, require_btc_confirm=cfg_d.require_btc_1h_confirm)
    train = backtest(arr, cfg_d, leverage=eff_lev, require_btc_confirm=cfg_d.require_btc_1h_confirm, i_start=52, i_end=mid)
    test = backtest(arr, cfg_d, leverage=eff_lev, require_btc_confirm=cfg_d.require_btc_1h_confirm, i_start=mid, i_end=n)

    def stats(trades):
        if not trades: return dict(n=0, pnl=0, wr=0, pf=None, mdd=0)
        pnls = np.array([t['pnl'] for t in trades])
        wins = pnls[pnls > 0]; losses = pnls[pnls <= 0]
        wr = 100 * len(wins)/len(pnls)
        pf = wins.sum() / abs(losses.sum()) if len(losses) and losses.sum() != 0 else None
        eq = np.cumsum(pnls) + 10000
        peaks = np.maximum.accumulate(eq)
        dd = ((eq - peaks) / peaks * 100).min() if len(eq) else 0
        return dict(n=len(pnls), pnl=float(pnls.sum()), wr=wr, pf=pf, mdd=abs(float(dd)))

    results[sym] = {
        'cfg_summary': f"{cfg_d.entry_type} {cfg_d.direction} | 1h {cfg_d.trend_filter_1h} | 4h {'ON' if cfg_d.require_4h_agreement else 'off'} | BTC-confirm {'ON' if cfg_d.require_btc_1h_confirm else 'off'}",
        'full': stats(full), 'train': stats(train), 'test': stats(test),
        'trades': full,
    }
    for t in full: t['symbol'] = sym
    all_trades.extend(full)
    print(f"  {sym:<10} n={len(full):>3} P&L=${stats(full)['pnl']:>+8.0f}  WR={stats(full)['wr']:.1f}%  "
          f"PF={stats(full)['pf']:.2f}" if stats(full)['pf'] else f"  {sym}: {len(full)} trades")

trades_df = pd.DataFrame(all_trades)
print(f'\nTotal: {len(trades_df)} trades, ${trades_df["pnl"].sum():+.2f}')

## 2 — Per-symbol metrics table

In [ ]:
rows = []
for sym, r in results.items():
    f = r['full']; tr = r['train']; te = r['test']
    rows.append({
        'symbol': sym, 'config': r['cfg_summary'],
        'trades': f['n'], 'pnl': round(f['pnl'], 0), 'wr': round(f['wr'], 1),
        'pf': round(f['pf'], 2) if f['pf'] else None,
        'mdd_pct': round(f['mdd'], 2),
        'train_pnl': round(tr['pnl'], 0), 'train_pf': round(tr['pf'], 2) if tr['pf'] else None,
        'test_pnl': round(te['pnl'], 0), 'test_pf': round(te['pf'], 2) if te['pf'] else None,
        'oos_holds': '✅' if (te['n'] >= 5 and te['pnl'] >= 0) else ('⚠️' if te['n'] >= 5 else '—'),
    })
df_summary = pd.DataFrame(rows).set_index('symbol')
df_summary

In [ ]:
# Per-symbol P&L bar chart (full sample)
fig, ax = plt.subplots(figsize=(13, 5))
bars = df_summary['pnl'].values
colors = ['#34d399' if x >= 0 else '#ef4444' for x in bars]
ax.bar(df_summary.index, bars, color=colors, edgecolor='#1b2420')
ax.axhline(0, color='#7a8480', linewidth=0.5)
for i, (sym, p) in enumerate(zip(df_summary.index, bars)):
    wr = df_summary.loc[sym, 'wr']; n = df_summary.loc[sym, 'trades']
    ax.annotate(f'${p:+.0f}\n{int(n)}t · {wr:.0f}%WR', (i, p),
                ha='center', va='bottom' if p >= 0 else 'top', fontsize=9, color='#e6eeea')
ax.set_ylabel('P&L ($)'); ax.set_title('Per-symbol P&L — full sample (deployed configs)')
plt.tight_layout(); plt.show()

## 3 — Aggregate equity curve

In [ ]:
# Sort all trades by timestamp, build cumulative equity curve (start at $10,000)
trades_df['ts'] = pd.to_datetime(trades_df['ts'])
trades_df = trades_df.sort_values('ts').reset_index(drop=True)
trades_df['cum_pnl'] = trades_df['pnl'].cumsum()
trades_df['equity'] = 10000 + trades_df['cum_pnl']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax1.plot(trades_df['ts'], trades_df['equity'], color='#34d399', linewidth=1.8)
ax1.fill_between(trades_df['ts'], 10000, trades_df['equity'],
                 where=(trades_df['equity'] >= 10000), alpha=0.15, color='#34d399')
ax1.fill_between(trades_df['ts'], 10000, trades_df['equity'],
                 where=(trades_df['equity'] < 10000), alpha=0.15, color='#ef4444')
ax1.axhline(10000, color='#7a8480', linewidth=0.5, linestyle='--')
ax1.set_ylabel('Equity ($)')
ax1.set_title(f"Aggregate equity curve — {len(trades_df)} trades, total P&L ${trades_df['pnl'].sum():+.0f}")

# Drawdown underneath
peaks = trades_df['equity'].cummax()
dd = (trades_df['equity'] - peaks) / peaks * 100
ax2.fill_between(trades_df['ts'], 0, dd, color='#ef4444', alpha=0.4)
ax2.set_ylabel('Drawdown (%)'); ax2.set_xlabel('date')
ax2.set_title(f'Drawdown from peak — max {dd.min():.2f}%')
plt.tight_layout(); plt.show()

## 4 — Walk-forward train/test split

In [ ]:
# Per-symbol train vs test grouped bars
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(df_summary))
w = 0.4
ax.bar(x - w/2, df_summary['train_pnl'], w, label='Train (first half)', color='#60a5fa', edgecolor='#1b2420')
ax.bar(x + w/2, df_summary['test_pnl'], w, label='Test (second half)', color='#fbbf24', edgecolor='#1b2420')
ax.axhline(0, color='#7a8480', linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(df_summary.index)
ax.set_ylabel('P&L ($)'); ax.set_title('Walk-forward — train vs test P&L per symbol')
ax.legend()
plt.tight_layout(); plt.show()

train_total = df_summary['train_pnl'].sum()
test_total = df_summary['test_pnl'].sum()
print(f'Total train: ${train_total:+.0f}   Test: ${test_total:+.0f}   '
      f'(OOS = {test_total/train_total*100:.0f}% of train)' if train_total else '')

## 5 — Exit reason breakdown

In [ ]:
reason_counts = trades_df.groupby('reason').agg(n=('pnl', 'count'), total_pnl=('pnl', 'sum')).reset_index()
reason_counts['avg_pnl'] = reason_counts['total_pnl'] / reason_counts['n']
reason_counts = reason_counts.sort_values('n', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
color_map = {'tp1': '#34d399', 'tp2': '#10b981', 'tp3': '#059669',
             'sl': '#ef4444', 'trail_stop': '#fbbf24', 'max_hold': '#60a5fa'}
colors = [color_map.get(r, '#7a8480') for r in reason_counts['reason']]
ax1.bar(reason_counts['reason'], reason_counts['n'], color=colors, edgecolor='#1b2420')
ax1.set_ylabel('Trade count'); ax1.set_title('Exit reason — counts')
for i, (_, row) in enumerate(reason_counts.iterrows()):
    pct = row['n'] / len(trades_df) * 100
    ax1.annotate(f"{int(row['n'])}\n({pct:.1f}%)", (i, row['n']),
                 ha='center', va='bottom', fontsize=9)

ax2.bar(reason_counts['reason'], reason_counts['total_pnl'], color=colors, edgecolor='#1b2420')
ax2.axhline(0, color='#7a8480', linewidth=0.5)
ax2.set_ylabel('Total P&L ($)'); ax2.set_title('Exit reason — P&L contribution')
for i, (_, row) in enumerate(reason_counts.iterrows()):
    ax2.annotate(f"${row['total_pnl']:+.0f}", (i, row['total_pnl']),
                 ha='center', va='bottom' if row['total_pnl'] >= 0 else 'top', fontsize=9)
plt.tight_layout(); plt.show()
reason_counts.round(2)

## 6 — Regime split (BTC 1h UP vs DOWN at entry)

In [ ]:
# Map btc at entry -> regime
trades_df['regime'] = trades_df['btc'].map({1: 'BTC UP', -1: 'BTC DOWN', 0: 'flat'})
regime_stats = trades_df.groupby(['symbol', 'regime']).agg(n=('pnl', 'count'), pnl=('pnl', 'sum')).reset_index()

fig, ax = plt.subplots(figsize=(13, 5))
pivot = regime_stats.pivot(index='symbol', columns='regime', values='pnl').fillna(0)
pivot = pivot.reindex(SYMBOLS)
x = np.arange(len(pivot))
w = 0.25
ax.bar(x - w, pivot.get('BTC UP', 0), w, label='BTC UP', color='#34d399', edgecolor='#1b2420')
ax.bar(x, pivot.get('BTC DOWN', 0), w, label='BTC DOWN', color='#ef4444', edgecolor='#1b2420')
if 'flat' in pivot.columns:
    ax.bar(x + w, pivot.get('flat', 0), w, label='BTC flat', color='#7a8480', edgecolor='#1b2420')
ax.axhline(0, color='#7a8480', linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(pivot.index)
ax.set_ylabel('P&L ($)'); ax.set_title('Per-symbol P&L by BTC 1h regime at entry')
ax.legend()
plt.tight_layout(); plt.show()

print('\nRegime totals:')
for reg in trades_df['regime'].dropna().unique():
    sub = trades_df[trades_df['regime'] == reg]
    print(f'  {reg:<10}  n={len(sub):>3}  P&L=${sub["pnl"].sum():+.0f}')

## 7 — Long vs Short split

In [ ]:
side_stats = trades_df.groupby(['symbol', 'side']).agg(n=('pnl', 'count'), pnl=('pnl', 'sum')).reset_index()
side_pivot = side_stats.pivot(index='symbol', columns='side', values='pnl').fillna(0).reindex(SYMBOLS)

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(side_pivot))
w = 0.4
ax.bar(x - w/2, side_pivot.get('long', 0), w, label='Longs', color='#60a5fa', edgecolor='#1b2420')
ax.bar(x + w/2, side_pivot.get('short', 0), w, label='Shorts', color='#ef4444', edgecolor='#1b2420')
ax.axhline(0, color='#7a8480', linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(side_pivot.index)
ax.set_ylabel('P&L ($)'); ax.set_title('Per-symbol P&L by side')
ax.legend()
plt.tight_layout(); plt.show()

total_long = trades_df[trades_df['side']=='long']['pnl'].sum()
total_short = trades_df[trades_df['side']=='short']['pnl'].sum()
n_long = (trades_df['side']=='long').sum()
n_short = (trades_df['side']=='short').sum()
print(f'Longs:  {n_long:>3} trades, P&L ${total_long:+.0f}')
print(f'Shorts: {n_short:>3} trades, P&L ${total_short:+.0f}')

## 8 — Headline numbers

In [ ]:
total_pnl = trades_df['pnl'].sum()
n = len(trades_df)
wins = (trades_df['pnl'] > 0).sum()
losses = (trades_df['pnl'] < 0).sum()
wr = wins / n * 100 if n else 0
gw = trades_df[trades_df['pnl'] > 0]['pnl'].sum()
gl = abs(trades_df[trades_df['pnl'] < 0]['pnl'].sum())
pf = gw / gl if gl else None
peaks = trades_df['equity'].cummax()
dd = ((trades_df['equity'] - peaks) / peaks * 100).min()

print('=' * 55)
print('  FULL-SAMPLE HEADLINE (9 active symbols, deployed)')
print('=' * 55)
print(f'  Trades:         {n}')
print(f'  Total P&L:      ${total_pnl:+,.2f}')
print(f'  Win rate:       {wr:.1f}%  ({wins}W / {losses}L)')
print(f'  Profit factor:  {pf:.2f}' if pf else '  Profit factor:  ∞')
print(f'  Gross wins:     ${gw:+,.2f}')
print(f'  Gross losses:   ${-gl:+,.2f}')
print(f'  Max drawdown:   {dd:.2f}%')
print(f'  Return on $10k: {total_pnl/10000*100:+.2f}%')